# Credit Card Fraud Detection — Feature Selection

## Objetivo

Este notebook contiene el proceso de selección y validación de features
para el modelo de detección de fraude.

El objetivo de esta etapa no es optimizar hiperparámetros ni comparar
diferentes algoritmos, sino determinar qué información debe formar parte
del modelo de producción.

Se utilizará XGBoost como modelo fijo para comparar diferentes conjuntos
de features bajo las mismas condiciones experimentales.

### Principios del experimento

- Mismo dataset
- Mismo train/test split
- Mismo modelo
- Mismos hiperparámetros
- Mismo random state
- Cambiar únicamente el conjunto de features

De esta manera, las diferencias observadas pueden atribuirse al conjunto
de variables utilizado.

---

### Feature engineering

Se consideran las siguientes transformaciones:

- `dob` → `age`
- `trans_date_trans_time` → `hour`
- ubicación del cliente + ubicación del comercio → `distance_km`

---

### Métricas

Debido al fuerte desbalance de clases, no se utilizará ROC-AUC como única
métrica.

Se registrarán:

- ROC-AUC
- PR-AUC
- Precision
- Recall
- F1-Score

Las métricas de clasificación se evaluarán con un threshold de 0.5 y,
posteriormente, se analizará el threshold utilizado en el modelo final.

In [1]:
# =========================
# 2. IMPORTS Y CONFIGURACIÓN
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack, csr_matrix
from sklearn.pipeline import Pipeline


from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("Set2")
plt.rcParams["figure.figsize"] = (12, 6)

RANDOM_STATE = 42

print("=" * 80)
print("CREDIT CARD FRAUD DETECTION - FEATURE SELECTION")
print("=" * 80)

CREDIT CARD FRAUD DETECTION - FEATURE SELECTION


## 3. Carga del dataset

Se utiliza el dataset procesado generado durante la etapa de EDA.

En esta etapa todavía no se seleccionan features. Primero cargamos y
verificamos la estructura de los datos.

In [2]:
# =========================
# 3. CARGA DE DATOS
# =========================

DATA_PATH = (
    r"C:\Users\Geronimo\Desktop\Credit-card-fraud-detection"
    r"\data\processed\fraud_clean.csv"
)

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")

Dataset shape: (1296675, 25)


## 4. Inspección inicial

Antes de realizar transformaciones verificamos:

- dimensiones
- valores nulos
- tipos de datos
- distribución del target

La variable objetivo es `is_fraud`.

In [3]:
# =========================
# 4. INSPECCIÓN INICIAL
# =========================

print("\n📊 DIMENSIONES:")
print(df.shape)

print("\n🔍 VALORES NULOS:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

print("\n📋 TIPOS DE DATOS:")
print(df.dtypes)

print("\n🎯 DISTRIBUCIÓN DEL TARGET:")
print(df["is_fraud"].value_counts())

print("\n🎯 DISTRIBUCIÓN RELATIVA:")
print(df["is_fraud"].value_counts(normalize=True))


📊 DIMENSIONES:
(1296675, 25)

🔍 VALORES NULOS:
Unnamed: 0               0
trans_date_trans_time    0
cc_num                   0
merchant                 0
category                 0
amt                      0
first                    0
last                     0
gender                   0
street                   0
city                     0
state                    0
zip                      0
lat                      0
long                     0
city_pop                 0
job                      0
dob                      0
trans_num                0
unix_time                0
dtype: int64

📋 TIPOS DE DATOS:
Unnamed: 0                 int64
trans_date_trans_time        str
cc_num                     int64
merchant                     str
category                     str
amt                      float64
first                        str
last                         str
gender                       str
street                       str
city                         str
state            

## 5. Feature Engineering

En esta etapa construimos las variables derivadas que pueden utilizarse
en el modelo.

### Edad

`dob` se transforma en `age`.

La edad se calcula respecto al año de la transacción y no mediante un año
fijo, evitando que la variable quede desactualizada al utilizar el modelo
en producción.

### Hora

La hora se obtiene de `trans_date_trans_time` cuando esta variable está
disponible.

### Distancia

`distance_km` representa la distancia entre la ubicación del cliente y
la ubicación del comercio.

Estas transformaciones se realizan antes de seleccionar el conjunto final
de features.

In [4]:
# =========================
# 5. FEATURE ENGINEERING
# =========================

df_model = df.copy()

# ---------------------------------
# 5.1 Fecha/hora de la transacción
# ---------------------------------

if "trans_date_trans_time" in df_model.columns:
    df_model["trans_date_trans_time"] = pd.to_datetime(
        df_model["trans_date_trans_time"]
    )

    # Extraer hora si todavía no existe
    if "hour" not in df_model.columns:
        df_model["hour"] = df_model["trans_date_trans_time"].dt.hour


# ---------------------------------
# 5.2 Edad
# ---------------------------------

if "dob" in df_model.columns:

    df_model["dob"] = pd.to_datetime(df_model["dob"])

    if "trans_date_trans_time" in df_model.columns:
        transaction_year = df_model["trans_date_trans_time"].dt.year
    else:
        # Fallback únicamente para este dataset procesado
        transaction_year = 2019

    df_model["age"] = (
        transaction_year - df_model["dob"].dt.year
    )

    df_model.drop(columns=["dob"], inplace=True)


# ---------------------------------
# 5.3 Eliminar identificadores
# ---------------------------------

drop_cols = [
    "Unnamed: 0",
    "trans_num",
    "first",
    "last",
    "street",
    "cc_num",
    "unix_time"
]

drop_cols = [
    col for col in drop_cols
    if col in df_model.columns
]

df_model.drop(columns=drop_cols, inplace=True)

print("Columnas eliminadas:")
print(drop_cols)

print("\nFeatures disponibles:")
print(df_model.columns.tolist())

Columnas eliminadas:
['Unnamed: 0', 'trans_num', 'first', 'last', 'street', 'cc_num', 'unix_time']

Features disponibles:
['trans_date_trans_time', 'merchant', 'category', 'amt', 'gender', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'merch_lat', 'merch_long', 'is_fraud', 'hour', 'distance_km', 'age']


## 6. Feature Set Candidato

Después de la revisión conceptual de las variables, definimos un conjunto
de features candidatas.

Las variables se organizan por bloques conceptuales.

### Núcleo transaccional

- `amt`
- `category`
- `merchant`
- `hour`

### Perfil del cliente

- `age`
- `gender`
- `job`

### Geografía

- `lat`
- `long`
- `merch_lat`
- `merch_long`
- `distance_km`
- `city`
- `state`
- `zip`
- `city_pop`

Las variables que fueron identificadas como identificadores o información
personal no adecuada para utilizar directamente como feature fueron
eliminadas previamente.

In [5]:
# =========================
# 6. FEATURE SET CANDIDATO
# =========================

FEATURE_GROUPS = {

    "core": [
        "amt",
        "category",
        "merchant",
        "hour"
    ],

    "profile": [
        "age",
        "gender",
        "job"
    ],

    "geo_coordinates": [
        "lat",
        "long",
        "merch_lat",
        "merch_long"
    ],

    "geo_context": [
        "city",
        "state",
        "zip",
        "city_pop"
    ],

    "geo_distance": [
        "distance_km"
    ]
}

# Verificar qué features existen realmente
for group, features in FEATURE_GROUPS.items():

    available = [
        feature
        for feature in features
        if feature in df_model.columns
    ]

    missing = [
        feature
        for feature in features
        if feature not in df_model.columns
    ]

    print(f"\n{group.upper()}")
    print(f"Disponibles: {available}")

    if missing:
        print(f"No encontradas: {missing}")


CORE
Disponibles: ['amt', 'category', 'merchant', 'hour']

PROFILE
Disponibles: ['age', 'gender', 'job']

GEO_COORDINATES
Disponibles: ['lat', 'long', 'merch_lat', 'merch_long']

GEO_CONTEXT
Disponibles: ['city', 'state', 'zip', 'city_pop']

GEO_DISTANCE
Disponibles: ['distance_km']


## 7. Train / Validation / Test Split

Se utiliza un único split estratificado para todos los experimentos.

El dataset se divide en:

- 70% entrenamiento
- 15% validación
- 15% test

El conjunto de entrenamiento se utiliza para ajustar los modelos.

El conjunto de validación se utiliza para comparar conjuntos de features
y seleccionar el threshold.

El conjunto de test permanece separado y se utiliza únicamente para la
evaluación final del modelo.

In [6]:
# =========================
# 7. TRAIN / VALIDATION / TEST SPLIT
# =========================

y = df_model["is_fraud"]
X = df_model.drop(columns=["is_fraud"])

# 70% train / 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

# 15% validation / 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print(f"Train:      {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")

print("\nFraud rate:")
print(f"Train:      {y_train.mean() * 100:.3f}%")
print(f"Validation: {y_val.mean() * 100:.3f}%")
print(f"Test:       {y_test.mean() * 100:.3f}%")


Train:      (907672, 17)
Validation: (194501, 17)
Test:       (194502, 17)

Fraud rate:
Train:      0.579%
Validation: 0.579%
Test:       0.579%


## 8. Preprocesamiento

Utilizamos `ColumnTransformer` junto con `OneHotEncoder` para separar
y transformar las variables categóricas y numéricas.

El encoder utiliza `handle_unknown="ignore"` para poder procesar
categorías que no hayan aparecido durante el entrenamiento.

El preprocesador se ajusta exclusivamente sobre el conjunto de
entrenamiento y luego se aplica a validation y test.

Este mismo preprocesamiento quedará integrado dentro del pipeline final
para garantizar que el modelo y las transformaciones se mantengan
consistentes durante la inferencia.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


def create_preprocessor(X_train, features):

    X_train_exp = X_train[features].copy()

    categorical_cols = X_train_exp.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    numeric_cols = [
        col for col in features
        if col not in categorical_cols
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                ),
                categorical_cols
            ),
            (
                "numeric",
                "passthrough",
                numeric_cols
            )
        ]
    )

    return preprocessor

## 9. Configuración del experimento

In [8]:

XGB_PARAMS = {
    "n_estimators": 50,
    "max_depth": 5,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "scale_pos_weight": 7,
    "tree_method": "hist",
    "random_state": RANDOM_STATE,
    "eval_metric": "logloss",
    "n_jobs": -1
}


def create_xgb_model():
    return XGBClassifier(**XGB_PARAMS)

In [9]:
def run_experiment(name, features):

    print("\n" + "=" * 80)
    print(f"EXPERIMENTO: {name}")
    print("=" * 80)

    print("\nFeatures:")
    print(features)

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                create_preprocessor(X_train, features)
            ),
            (
                "model",
                create_xgb_model()
            )
        ]
    )

    pipeline.fit(
        X_train[features],
        y_train
    )

    y_val_proba = pipeline.predict_proba(
        X_val[features]
    )[:, 1]

    y_val_pred = (
        y_val_proba >= 0.5
    ).astype(int)

    encoded_features = pipeline.named_steps[
        "preprocessor"
    ].transform(
        X_train[features]
    ).shape[1]

    results = {
        "Experiment": name,
        "Features": len(features),
        "Encoded Features": encoded_features,
        "ROC-AUC": roc_auc_score(
            y_val,
            y_val_proba
        ),
        "PR-AUC": average_precision_score(
            y_val,
            y_val_proba
        ),
        "Precision": precision_score(
            y_val,
            y_val_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_val,
            y_val_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_val,
            y_val_pred,
            zero_division=0
        )
    }

    print("\nValidation metrics:")

    for metric, value in results.items():

        if isinstance(value, float):
            print(f"{metric}: {value:.4f}")

        else:
            print(f"{metric}: {value}")

    return {
        "name": name,
        "features": features,
        "pipeline": pipeline,
        "y_val_proba": y_val_proba,
        "results": results
    }

## 10. Experimento A — Baseline

El baseline utiliza todas las features candidatas.

Este resultado servirá como referencia para determinar qué información
puede eliminarse sin producir una pérdida relevante de rendimiento.

In [10]:
# =========================
# 10. BASELINE
# =========================

ALL_FEATURES = (
    FEATURE_GROUPS["core"]
    + FEATURE_GROUPS["profile"]
    + FEATURE_GROUPS["geo_coordinates"]
    + FEATURE_GROUPS["geo_context"]
    + FEATURE_GROUPS["geo_distance"]
)

ALL_FEATURES = [
    feature
    for feature in ALL_FEATURES
    if feature in X_train.columns
]

experiment_baseline = run_experiment(
    "A - Baseline",
    ALL_FEATURES
)


EXPERIMENTO: A - Baseline

Features:
['amt', 'category', 'merchant', 'hour', 'age', 'gender', 'job', 'lat', 'long', 'merch_lat', 'merch_long', 'city', 'state', 'zip', 'city_pop', 'distance_km']

Validation metrics:
Experiment: A - Baseline
Features: 16
Encoded Features: 2158
ROC-AUC: 0.9957
PR-AUC: 0.8690
Precision: 0.8303
Recall: 0.8126
F1: 0.8214


## 11. Experimento B — Núcleo transaccional

Este experimento utiliza únicamente las variables directamente
relacionadas con la transacción y las features derivadas consideradas
fundamentales.

Incluye:

- monto
- categoría
- comercio
- hora
- edad
- distancia

In [11]:
# =========================
# 11. NÚCLEO
# =========================

CORE_FEATURES = [
    "amt",
    "category",
    "merchant",
    "hour",
    "age",
    "distance_km"
]

CORE_FEATURES = [
    feature
    for feature in CORE_FEATURES
    if feature in X_train.columns
]

experiment_core = run_experiment(
    "B - Core",
    CORE_FEATURES
)


EXPERIMENTO: B - Core

Features:
['amt', 'category', 'merchant', 'hour', 'age', 'distance_km']

Validation metrics:
Experiment: B - Core
Features: 6
Encoded Features: 711
ROC-AUC: 0.9941
PR-AUC: 0.8652
Precision: 0.8149
Recall: 0.8055
F1: 0.8102


## 12. Experimento C — Núcleo + Perfil

Se agregan las variables relacionadas con el perfil del cliente:

- `gender`
- `job`

El objetivo es medir cuánto aporta este bloque respecto al núcleo.

In [12]:
# =========================
# 12. CORE + PROFILE
# =========================

CORE_PROFILE_FEATURES = (
    CORE_FEATURES
    + FEATURE_GROUPS["profile"]
)

CORE_PROFILE_FEATURES = list(dict.fromkeys([
    feature
    for feature in CORE_PROFILE_FEATURES
    if feature in X_train.columns
]))

experiment_core_profile = run_experiment(
    "C - Core + Profile",
    CORE_PROFILE_FEATURES
)


EXPERIMENTO: C - Core + Profile

Features:
['amt', 'category', 'merchant', 'hour', 'age', 'distance_km', 'gender', 'job']

Validation metrics:
Experiment: C - Core + Profile
Features: 8
Encoded Features: 1207
ROC-AUC: 0.9950
PR-AUC: 0.8689
Precision: 0.7929
Recall: 0.8091
F1: 0.8009


## 13. Experimento D — Núcleo + Geografía absoluta

Se agregan las coordenadas del cliente y del comercio:

- `lat`
- `long`
- `merch_lat`
- `merch_long`

El objetivo es comprobar si conocer las ubicaciones absolutas aporta
información adicional respecto de `distance_km`.

In [13]:
# =========================
# 13. CORE + COORDINATES
# =========================

CORE_COORD_FEATURES = (
    CORE_FEATURES
    + FEATURE_GROUPS["geo_coordinates"]
)

CORE_COORD_FEATURES = list(dict.fromkeys([
    feature
    for feature in CORE_COORD_FEATURES
    if feature in X_train.columns
]))

experiment_core_coordinates = run_experiment(
    "D - Core + Coordinates",
    CORE_COORD_FEATURES
)


EXPERIMENTO: D - Core + Coordinates

Features:
['amt', 'category', 'merchant', 'hour', 'age', 'distance_km', 'lat', 'long', 'merch_lat', 'merch_long']

Validation metrics:
Experiment: D - Core + Coordinates
Features: 10
Encoded Features: 715
ROC-AUC: 0.9936
PR-AUC: 0.8619
Precision: 0.8093
Recall: 0.8028
F1: 0.8061


## 14. Experimento E — Núcleo + Geografía categórica/contextual

Se agregan:

- `city`
- `state`
- `zip`
- `city_pop`

El objetivo es medir si la información geográfica categórica aporta
información adicional al modelo.

In [14]:
# =========================
# 14. CORE + GEO CONTEXT
# =========================

CORE_GEO_CONTEXT_FEATURES = (
    CORE_FEATURES
    + FEATURE_GROUPS["geo_context"]
)

CORE_GEO_CONTEXT_FEATURES = list(dict.fromkeys([
    feature
    for feature in CORE_GEO_CONTEXT_FEATURES
    if feature in X_train.columns
]))

experiment_core_geo_context = run_experiment(
    "E - Core + Geographic Context",
    CORE_GEO_CONTEXT_FEATURES
)


EXPERIMENTO: E - Core + Geographic Context

Features:
['amt', 'category', 'merchant', 'hour', 'age', 'distance_km', 'city', 'state', 'zip', 'city_pop']

Validation metrics:
Experiment: E - Core + Geographic Context
Features: 10
Encoded Features: 1658
ROC-AUC: 0.9935
PR-AUC: 0.8636
Precision: 0.8059
Recall: 0.8037
F1: 0.8048


## 15. Experimento F — Núcleo + Geografía completa

Se incorporan todas las variables geográficas candidatas.

Este experimento permite evaluar si la combinación completa aporta
información adicional respecto de las representaciones parciales.

In [15]:
# =========================
# 15. CORE + FULL GEO
# =========================

FULL_GEO_FEATURES = (
    CORE_FEATURES
    + FEATURE_GROUPS["geo_coordinates"]
    + FEATURE_GROUPS["geo_context"]
    + FEATURE_GROUPS["geo_distance"]
)

FULL_GEO_FEATURES = list(dict.fromkeys([
    feature
    for feature in FULL_GEO_FEATURES
    if feature in X_train.columns
]))

experiment_full_geo = run_experiment(
    "F - Core + Full Geography",
    FULL_GEO_FEATURES
)


EXPERIMENTO: F - Core + Full Geography

Features:
['amt', 'category', 'merchant', 'hour', 'age', 'distance_km', 'lat', 'long', 'merch_lat', 'merch_long', 'city', 'state', 'zip', 'city_pop']

Validation metrics:
Experiment: F - Core + Full Geography
Features: 14
Encoded Features: 1662
ROC-AUC: 0.9934
PR-AUC: 0.8662
Precision: 0.8073
Recall: 0.8073
F1: 0.8073


## 16. Comparación de experimentos

Comparamos los diferentes conjuntos de features utilizando exactamente
el mismo modelo y el mismo conjunto de validation.

El conjunto de test se mantiene completamente separado y no se utiliza
para seleccionar features ni thresholds.

In [16]:
# =========================
# 16. COMPARACIÓN
# =========================

experiments = [
    experiment_baseline,
    experiment_core,
    experiment_core_profile,
    experiment_core_coordinates,
    experiment_core_geo_context,
    experiment_full_geo
]

results_df = pd.DataFrame([
    experiment["results"]
    for experiment in experiments
])

results_df = results_df.sort_values(
    by="PR-AUC",
    ascending=False
).reset_index(drop=True)

results_df

,Experiment,Features,Encoded Features,ROC-AUC,PR-AUC,Precision,Recall,F1
0,A - Baseline,16,2158,0.995667,0.869007,0.830309,0.812611,0.821364
1,C - Core + Profile,8,1207,0.995008,0.868938,0.792863,0.809059,0.800879
2,F - Core + Full Geography,14,1662,0.993355,0.866170,0.807282,0.807282,0.807282
3,B - Core,6,711,0.994083,0.865218,0.814915,0.805506,0.810183
4,E - Core + Geographic Context,10,1658,0.993485,0.863554,0.805877,0.803730,0.804802
5,D - Core + Coordinates,10,715,0.993567,0.861920,0.809311,0.802842,0.806063


## 17. Pipeline C — XGBoost


In [17]:
C_FEATURES = [
    "amt",
    "category",
    "merchant",
    "hour",
    "age",
    "gender",
    "job",
    "distance_km"
]

In [18]:
model_pipeline_C = Pipeline(
    steps=[
        ("preprocessor", create_preprocessor(X_train, C_FEATURES)),
        ("model", create_xgb_model())
    ]
)

In [19]:
model_pipeline_C.fit(
    X_train[C_FEATURES],
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['amt','category','merchant',...,'gender','job','distance_km']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all

In [20]:
y_val_prob_C = model_pipeline_C.predict_proba(
    X_val[C_FEATURES]
)[:, 1]

print("Predictions generated:", len(y_val_prob_C))

Predictions generated: 194501


## 18. Selección del threshold

In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.5, 0.55, 0.6, 0.7, 0.8, 0.9]

results = []

for threshold in thresholds:
    y_val_pred = (y_val_prob_C >= threshold).astype(int)

    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)

    results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

pd.DataFrame(results)

,threshold,precision,recall,f1
0,0.50,0.792863,0.809059,0.800879
1,0.55,0.829268,0.785080,0.806569
2,0.60,0.844143,0.774423,0.807781
3,0.70,0.859127,0.769094,0.811621
4,0.80,0.933103,0.718472,0.811841
5,0.90,0.983459,0.580817,0.730318


In [22]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.50, 0.91, 0.01)

threshold_results = []

for threshold in thresholds:
    y_val_pred = (y_val_prob_C >= threshold).astype(int)

    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)

    threshold_results.append({
        "threshold": round(threshold, 2),
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.sort_values(
    "f1",
    ascending=False
).head(10)

,threshold,precision,recall,f1
25,0.75,0.880829,0.754885,0.813008
24,0.74,0.871066,0.761989,0.812885
23,0.73,0.866332,0.765542,0.812824
22,0.72,0.861692,0.769094,0.812764
21,0.71,0.860835,0.769094,0.812383
18,0.68,0.859267,0.769982,0.812178
26,0.76,0.882292,0.752220,0.812081
30,0.80,0.933103,0.718472,0.811841
20,0.70,0.859127,0.769094,0.811621
19,0.69,0.859127,0.769094,0.811621


In [23]:
THRESHOLD_C = 0.75

print(
    f"Selected threshold: {THRESHOLD_C}"
)

Selected threshold: 0.75


## 19. Entrenamiento del modelo final

In [24]:
# Combinar Train + Validation para el entrenamiento final

X_train_final = pd.concat(
    [
        X_train[C_FEATURES],
        X_val[C_FEATURES]
    ],
    axis=0
)

y_train_final = pd.concat(
    [
        y_train,
        y_val
    ],
    axis=0
)

print("Final training set:", X_train_final.shape)
print("Final target:", y_train_final.shape)
print(f"Fraud rate: {y_train_final.mean():.3%}")

Final training set: (1102173, 8)
Final target: (1102173,)
Fraud rate: 0.579%


In [25]:
final_pipeline_C = Pipeline(
    steps=[
        (
            "preprocessor",
            create_preprocessor(
                X_train_final,
                C_FEATURES
            )
        ),
        (
            "model",
            create_xgb_model()
        )
    ]
)

In [26]:
final_pipeline_C.fit(
    X_train_final,
    y_train_final
)

print("Final model trained successfully.")

Final model trained successfully.


## 20. Evaluación final sobre Test

In [27]:
y_test_prob_C = final_pipeline_C.predict_proba(
    X_test[C_FEATURES]
)[:, 1]

print("Test predictions generated:", len(y_test_prob_C))

Test predictions generated: 194502


In [28]:
y_test_pred_C = (
    y_test_prob_C >= THRESHOLD_C
).astype(int)

In [29]:
test_metrics_C = {
    "ROC-AUC": roc_auc_score(y_test, y_test_prob_C),
    "PR-AUC": average_precision_score(y_test, y_test_prob_C),
    "Precision": precision_score(
        y_test,
        y_test_pred_C,
        zero_division=0
    ),
    "Recall": recall_score(
        y_test,
        y_test_pred_C,
        zero_division=0
    ),
    "F1": f1_score(
        y_test,
        y_test_pred_C,
        zero_division=0
    )
}

print("Final Test Metrics:")
for metric, value in test_metrics_C.items():
    print(f"{metric}: {value:.4f}")

Final Test Metrics:
ROC-AUC: 0.9958
PR-AUC: 0.8791
Precision: 0.8739
Recall: 0.7815
F1: 0.8251


## 21. Análisis de errores

In [30]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_test,
    y_test_pred_C
)

tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)

print("\nTrue Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

Confusion Matrix:
[[193249    127]
 [   246    880]]

True Negatives: 193249
False Positives: 127
False Negatives: 246
True Positives: 880


In [31]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_test,
        y_test_pred_C,
        target_names=["Legitimate", "Fraud"],
        digits=4
    )
)

              precision    recall  f1-score   support

  Legitimate     0.9987    0.9993    0.9990    193376
       Fraud     0.8739    0.7815    0.8251      1126

    accuracy                         0.9981    194502
   macro avg     0.9363    0.8904    0.9121    194502
weighted avg     0.9980    0.9981    0.9980    194502



## 22. Guardado del modelo

In [32]:
import joblib
from pathlib import Path

In [33]:
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_artifact = {
    "model": final_pipeline_C,
    "threshold": THRESHOLD_C,
    "features": C_FEATURES
}

MODEL_PATH = MODEL_DIR / "fraud_detection_pipeline.joblib"

joblib.dump(
    model_artifact,
    MODEL_PATH
)

print(f"Model saved to: {MODEL_PATH}")

Model saved to: models\fraud_detection_pipeline.joblib


## 23. Verificación del modelo guardado

In [34]:
loaded_artifact = joblib.load(MODEL_PATH)

loaded_model = loaded_artifact["model"]
loaded_threshold = loaded_artifact["threshold"]
loaded_features = loaded_artifact["features"]

print("Model loaded successfully.")
print("Threshold:", loaded_threshold)
print("Features:", loaded_features)

Model loaded successfully.
Threshold: 0.75
Features: ['amt', 'category', 'merchant', 'hour', 'age', 'gender', 'job', 'distance_km']


In [35]:
loaded_test_prob = loaded_model.predict_proba(
    X_test[loaded_features]
)[:, 1]

loaded_test_pred = (
    loaded_test_prob >= loaded_threshold
).astype(int)

print("Predictions generated:", len(loaded_test_pred))
print("Predictions match original:",
      np.array_equal(loaded_test_pred, y_test_pred_C))

Predictions generated: 194502
Predictions match original: True


## 24. Categorías para la aplicación

In [36]:
import pandas as pd

df_app = pd.read_csv("C:\\Users\\Geronimo\\Desktop\\Credit-card-fraud-detection\\data\\processed\\fraud_clean.csv")

categorical_features = [
    "category",
    "merchant",
    "gender",
    "job"
]

for col in categorical_features:
    print(f"\n{'=' * 50}")
    print(f"{col.upper()}")
    print(f"Valores únicos: {df_app[col].nunique()}")
    print(df_app[col].dropna().unique())


CATEGORY
Valores únicos: 14
<ArrowStringArray>
[      'misc_net',    'grocery_pos',  'entertainment',  'gas_transport',
       'misc_pos',    'grocery_net',   'shopping_net',   'shopping_pos',
    'food_dining',  'personal_care', 'health_fitness',         'travel',
      'kids_pets',           'home']
Length: 14, dtype: str

MERCHANT
Valores únicos: 693
<ArrowStringArray>
[         'fraud_Rippin, Kub and Mann',     'fraud_Heller, Gutmann and Zieme',
                'fraud_Lind-Buckridge',  'fraud_Kutch, Hermiston and Farrell',
                 'fraud_Keeling-Crist',    'fraud_Stroman, Hudson and Erdman',
               'fraud_Rowe-Vandervort',                'fraud_Corwin-Collins',
                    'fraud_Herzog Ltd',   'fraud_Schoen, Kuphal and Nitzsche',
 ...
  'fraud_Kohler, Lindgren and Koelpin',                'fraud_Heller-Abshire',
  'fraud_Swift, Bradtke and Marquardt',   'fraud_Larson, Quitzon and Spencer',
 'fraud_Kilback, Nitzsche and Leffler',              'fraud_Jakubo

In [37]:
print("=" * 50)
print("GENDER")
print("Valores únicos:", df_app["gender"].nunique())
print(df_app["gender"].dropna().unique())

print("=" * 50)
print("JOB")
print("Valores únicos:", df_app["job"].nunique())
print(df_app["job"].dropna().unique())

GENDER
Valores únicos: 2
<ArrowStringArray>
['F', 'M']
Length: 2, dtype: str
JOB
Valores únicos: 494
<ArrowStringArray>
[        'Psychologist, counselling', 'Special educational needs teacher',
       'Nature conservation officer',                   'Patent attorney',
    'Dance movement psychotherapist',                 'Transport planner',
                   'Arboriculturist',              'Designer, multimedia',
         'Public affairs consultant',                       'Pathologist',
 ...
                  'Industrial buyer',             'Accountant, chartered',
            'Air traffic controller',                   'Careers adviser',
               'Information officer',                       'Ship broker',
                   'Legal secretary',                         'Homeopath',
                         'Solicitor',                 'Warehouse manager']
Length: 494, dtype: str


In [38]:
import joblib
from pathlib import Path

MODEL_PATH = Path("../models/fraud_detection_pipeline.joblib")

artifact = joblib.load(MODEL_PATH)

pipeline = artifact["model"]
threshold = artifact["threshold"]
features = artifact["features"]

print("Threshold:", threshold)
print("Features:", features)

Threshold: 0.75
Features: ['amt', 'category', 'merchant', 'hour', 'age', 'gender', 'job', 'distance_km']


In [39]:
preprocessor = pipeline.named_steps["preprocessor"]

encoder = preprocessor.named_transformers_["categorical"]

categorical_columns = [
    "category",
    "merchant",
    "gender",
    "job"
]

category_values = {
    column: categories.tolist()
    for column, categories in zip(
        categorical_columns,
        encoder.categories_
    )
}

for column, values in category_values.items():
    print(f"{column}: {len(values)} valores")

category: 14 valores
merchant: 693 valores
gender: 2 valores
job: 494 valores


In [40]:
artifact["category_values"] = category_values

In [41]:
joblib.dump(artifact, MODEL_PATH)

print(f"Artifact actualizado: {MODEL_PATH}")

Artifact actualizado: ..\models\fraud_detection_pipeline.joblib


In [43]:
print(MODEL_PATH.resolve())

C:\Users\Geronimo\Desktop\Credit-card-fraud-detection\models\fraud_detection_pipeline.joblib


In [42]:
loaded_artifact = joblib.load(MODEL_PATH)

print("Keys del artifact:")
print(loaded_artifact.keys())

print("\nCategorías:")
for column, values in loaded_artifact["category_values"].items():
    print(f"{column}: {len(values)}")

Keys del artifact:
dict_keys(['model', 'threshold', 'features', 'category_values'])

Categorías:
category: 14
merchant: 693
gender: 2
job: 494
